# Qwen2.5-VL-3B: table reading vs. multi-step reasoning

Evaluates Qwen2.5-VL-3B (4-bit NF4, Colab T4) on "remove one song, re-rank the rest per judge, sum the scores",
and compares **pure VLM reasoning** with **VLM reads the table -> Python computes**.

Results and failure analysis: see `README.md` and `results.json`. Requires a T4 GPU runtime.

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow bitsandbytes

## 1. Load the model (4-bit NF4)

float32 needs about 13 GB and does not fit on a T4 (layers get offloaded to CPU, very slow).
4-bit NF4 needs about 2.4 GB and keeps everything on the GPU.
If the notebook was run before in the same runtime, restart the session first so old weights are freed.

In [ ]:
import torch, json, re, time, os
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id, quantization_config=bnb, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print("device:", model.device, f"| GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

def ask(image_path, question, max_new_tokens=128):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{image_path}", "max_pixels": 640 * 640},
        {"type": "text", "text": question},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    inputs = processor(text=[text], images=images, videos=videos, padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # greedy decoding
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0]

## 2. The "tool": deterministic ranking and summing

After removing one song, each judge re-ranks the remaining songs by their original score
(best gets n points, worst gets 1), then scores are summed per song.

In [ ]:
def rerank_without(table, removed):
    songs = [s for s in table if s != removed]
    n = len(songs)
    totals = {s: 0 for s in songs}
    for j in range(len(table[songs[0]])):
        order = sorted(songs, key=lambda s: table[s][j], reverse=True)
        for rank, s in enumerate(order):
            totals[s] += n - rank
    return totals

## 3. Optional: probing a real lecture slide

The slide screenshot is not included in this repo (course material).
To run this section, upload your own table screenshot as `/content/screenshot1.png`.
Observations from the original run are summarized in `README.md`.

In [ ]:
SLIDE = "/content/screenshot1.png"
if os.path.exists(SLIDE):
    print(ask(SLIDE, "图中有哪些数字？请逐一列出。", 200))
    print(ask(SLIDE, "请用 JSON 格式输出标题文字的边界框坐标 bbox_2d。", 100))

    # Grounding coordinates live in the *resized* image space; rescale before drawing on the original.
    from PIL import Image, ImageDraw
    from qwen_vl_utils.vision_process import smart_resize
    img = Image.open(SLIDE).convert("RGB")
    W, H = img.size
    h, w = smart_resize(H, W, factor=28, max_pixels=640 * 640)
    print("original:", W, H, "| model sees:", w, h)
    x1, y1, x2, y2 = 0, 14, 156, 30      # replace with the bbox_2d printed above
    sx, sy = W / w, H / h
    ImageDraw.Draw(img).rectangle([x1 * sx, y1 * sy, x2 * sx, y2 * sy], outline="red", width=4)
    display(img)
else:
    print("screenshot1.png not found, skipping this section")

## 4. Synthetic test set

Random score tables (4-5 songs, 5-7 judges, each judge gives every song a different score, so no ties).
Ground truth is computed by code, so no hand-labelled answers. Fixed seed for reproducibility.

In [ ]:
import random
import matplotlib.pyplot as plt

os.makedirs("/content/eval", exist_ok=True)

def make_table(n_songs, n_judges, rng):
    names = [chr(65 + i) for i in range(n_songs)]
    table = {s: [] for s in names}
    for _ in range(n_judges):
        scores = list(range(1, n_songs + 1))
        rng.shuffle(scores)
        for s, v in zip(names, scores):
            table[s].append(v)
    return table

def draw_table(table, path):
    n_judges = len(next(iter(table.values())))
    cols = [f"J{j+1}" for j in range(n_judges)] + ["Tot."]
    cells = [[*v, sum(v)] for v in table.values()]
    fig, ax = plt.subplots(figsize=(1 + 0.8 * n_judges, 0.8 + 0.5 * len(table)))
    ax.axis("off")
    ax.table(cellText=cells, rowLabels=list(table), colLabels=cols, loc="center").scale(1, 1.8)
    plt.title("Song contest: judges' scores", pad=8)
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

rng = random.Random(42)
gen = []
for k in range(8):
    n_songs, n_judges = rng.choice([4, 5]), rng.choice([5, 6, 7])
    table = make_table(n_songs, n_judges, rng)
    path = f"/content/eval/g{k+1}.png"
    draw_table(table, path)

    removed = rng.choice(list(table))
    others = [s for s in table if s != removed]
    totals = rerank_without(table, removed)
    if k % 2 == 0:
        qtext = (f"如果取消歌曲{removed}，剩下的歌曲每位评委重新按分数从高到低赋{len(others)}到1分"
                 f"（最高分得{len(others)}分），获胜歌曲的总分是多少？")
        answer = max(totals.values())
        solver = (lambda t, r=removed: max(rerank_without(t, r).values()))
    else:
        target = rng.choice(others)
        qtext = (f"如果取消歌曲{removed}，剩下的歌曲每位评委重新按分数从高到低赋{len(others)}到1分"
                 f"（最高分得{len(others)}分），歌曲{target}的总分是多少？")
        answer = totals[target]
        solver = (lambda t, r=removed, g=target: rerank_without(t, r)[g])

    gen.append({
        "image": path, "question": qtext,
        "extract_hint": f"{'、'.join(table)}共{len(table)}行，每行是J1到J{n_judges}共{n_judges}位评委的打分，不含Tot.列",
        "answer": answer, "solver": solver,
    })
    print(f"g{k+1}: {n_songs} songs, {n_judges} judges | {qtext} | answer={answer}")

## 5. Evaluation: pure VLM vs. VLM reads table + Python computes

- **pure**: the model reasons step by step and ends with `最终答案：<number>`.
- **tool**: the model only converts the table image to JSON; `rerank_without` does the ranking and summing.

The answer parser accepts variants like "最终答案是17" (the first version missed these and mis-scored them as failures).
If no final answer is found within 500 tokens, the question is retried once with 1000 tokens.

In [ ]:
questions = []
# The original run also included one real lecture-slide screenshot as q1.png (not in this repo).
if os.path.exists("/content/eval/q1.png"):
    questions.append({
        "image": "/content/eval/q1.png",
        "question": "如果取消歌曲B，剩下的歌曲A、C、D每位评委重新按分数从高到低赋3、2、1分，获胜歌曲的总分是多少？",
        "extract_hint": "第一张表格，A、B、C、D四行，每行是J1到J7七位评委的打分，不含Tot.列",
        "answer": 18,
        "solver": lambda t: max(rerank_without(t, "B").values()),
    })
questions += gen

def parse_final(text):
    m = re.findall(r"最终答案\D{0,4}?(-?\d+(?:\.\d+)?)", text)
    return float(m[-1]) if m else None

def run_pure(q):
    prompt = q["question"] + " 请先写出推理步骤，最后一行以“最终答案：”开头只写一个数字。"
    out = ask(q["image"], prompt, 500)
    p = parse_final(out)
    if p is None:
        out = ask(q["image"], prompt, 1000)
        p = parse_final(out)
    return p, out

def run_tool(q):
    prompt = (f"把图中的表格转成JSON：键是行名，值是该行从左到右的数字列表。{q['extract_hint']}。"
              "只输出JSON，不要其他文字。")
    raw = ask(q["image"], prompt, 300)
    try:
        table = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
        return float(q["solver"](table)), raw
    except Exception as e:
        return None, f"ERROR {e} | {raw}"

results = []
for i, q in enumerate(questions):
    t0 = time.time()
    p, p_out = run_pure(q)
    t, t_out = run_tool(q)
    ok_p = p is not None and abs(p - q["answer"]) < 1e-6
    ok_t = t is not None and abs(t - q["answer"]) < 1e-6
    results.append({"i": i, "answer": q["answer"], "pure": p, "pure_ok": ok_p,
                    "tool": t, "tool_ok": ok_t, "pure_out": p_out, "tool_out": t_out})
    print(f"Q{i+1}: answer={q['answer']} | pure={p} {'OK' if ok_p else 'X'} | "
          f"tool={t} {'OK' if ok_t else 'X'} | {time.time()-t0:.0f}s")

n = len(results)
print(f"\npure VLM: {sum(r['pure_ok'] for r in results)}/{n}   VLM+Python: {sum(r['tool_ok'] for r in results)}/{n}")
json.dump(results, open("/content/results.json", "w"), ensure_ascii=False, indent=2)